In [19]:
import warnings
warnings.simplefilter('ignore')

%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly
import plotly.graph_objs as go

init_notebook_mode(connected=True)

from sklearn.metrics import mean_squared_error as mse
from scipy.stats import linregress
from scipy.interpolate import UnivariateSpline
from datetime import datetime, timedelta

In [20]:
depths = [0,25,50,100,250]
def df_lake(d):
    df = pd.read_csv('Final_point_1_all_depths.csv')
    df.date = pd.to_datetime(df.date)
    df = df.groupby(['date', 'depth'])[['temp']].mean()
    df = df.reset_index()
    df = df.sort_values(by='date')
    df = df[df.depth == d]
    return df

def df_lake_mean(d, m1, m2):
    df = df_lake(d)
    df['year'] = df.date.apply(lambda x: x.year)
    df = df[(df.date.dt.month >= m1) & (df.date.dt.month <= m2)]
    df1 = df.groupby('year')[['temp']].mean()
    l = []
    for y in range(1948, 2024):
        df_for = df[df.year == y]
        if len(df_for) != 0:
            if max_length(df_for, y=y, m1=m1, m2=m2) > 20:
                l.append(np.nan)
            else:
                l.append(1)
    df1['flag'] = l
    df1 = df1.dropna()
    return df1

def df_lake_days(d):
    df = df_lake(d)
    df['year'] = df.date.apply(lambda x: x.year)
    df['day'] = df.date.apply(lambda x: x.dayofyear)
    df1 = df.groupby('year')[['temp']].max()
    l = []
    before_l = []
    next_l = []
    for y in range(1948, 2024):
        df_for = df[df.year == y]
        if max_length(df_for, y=y, m1=6, m2=11) > 20:
            if len(df_for) != 0:
                l.append(np.nan)
                before_l.append(np.nan)
                next_l.append(np.nan)
            continue
        before = None
        next = None
        flag = 0
        for index, row in df_for.iterrows():
            flag *= 2
            if (row.temp == df1[df1.index == y].temp.tolist()[0]) and (flag == 0):
                l.append(row.day)
                result = row.day
                if before is not None:
                    before_l.append(result - before)
                else:
                    before_l.append(np.nan)
                flag = 1
            before = row['day']
            next = row['day']
            if flag == 2:
                break
        if next is not None:
            next_l.append(next - result)
        else:
            next_l.append(np.nan)
    df1['day'] = l
    df1['before'] = before_l
    df1['next'] = next_l
    df1 = df1.dropna()
    return df1

cities = {'Улан-Удэ': 30823, 'Чита': 30758, 'Иркутск': 30710, 'Большое Голоустное': 30727, 'Бабушкин': 30822}
def df_city(city):
    header = ["ind", "year", "month", "day", "q_temp", 'min', 'mean', 'max', 'rain']
    df = pd.read_csv(f'{cities[city]}.txt', sep=";", engine='python', na_values=['     '], keep_default_na=False, header=None, names=header)
    df['date'] = df.apply(lambda row: datetime(int(row['year']), int(row['month']), int(row['day'])), axis=1)
    df = df[['date', 'mean']]
    df.columns = ['date', 'temp']
    df = df.dropna()
    df = df[(df.date.dt.year >= 1948) & (df.date.dt.year <= 2023)]
    df = df.sort_values(by='date')
    return df

def df_city_mean(city, m1, m2):
    df = df_city(city)
    df['year'] = df.date.apply(lambda x: x.year)
    df = df[(df.date.dt.month >= m1) & (df.date.dt.month <= m2)]
    df1 = df.groupby('year')[['temp']].mean()
    return df1

def df_city_days(city):
    df = df_city(city)
    df['year'] = df.date.apply(lambda x: x.year)
    df['day'] = df.date.apply(lambda x: x.dayofyear)
    df1 = df.groupby('year')[['temp']].max()
    l = []
    for y in range(1948, 2024):
        df_for = df[df.year == y]
        for index, row in df_for.iterrows():
            if row.temp == df1[df1.index == y].temp.tolist()[0]:
                l.append(row['day'])
                break
    df1['day'] = l
    return df1

def max_length(df, y, m1, m2):
    df1 = df[(df.date.dt.month >= m1) & (df.date.dt.month <= m2)]
    df1.loc[len(df1)] = {'date': datetime(y, m1, 1), 'depth': -1, 'temp': -1, 'day': -1, 'year': -1}
    df1.loc[len(df1)] = {'date': datetime(y, m2, 30), 'depth': -1, 'temp': -1, 'day': -1, 'year': -1}
    df1 = df1.sort_values(by='date')
    df1['next'] = df1.date.shift(-1)
    df1['diff'] = (df1.next - df1.date).dt.days
    return df1['diff'].max()

In [21]:
#Номер дня в году с максимальной температурой для всех глубин и метеостанций
tr = []
colors = ['red', 'blue', 'black', 'green', 'yellow']
c_ind = 0

for city in ['Иркутск', 'Улан-Удэ', 'Большое Голоустное', 'Бабушкин', 'Чита']:
    df1 = df_city_days(city)
    trace = go.Scatter(
        x=df1.index,
        y=df1.day,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'{city}'
    )
    p = np.poly1d(np.polyfit(df1.index, df1.day, 1))
    trace_line = go.Scatter(
        x=np.linspace(min(df1.index), max(df1.index), 1000),
        y=p(np.linspace(min(df1.index), max(df1.index), 1000)),
        mode='lines',
        line=dict(color=colors[c_ind], width=2),
        name=f'Линейная аппроксимация для {city}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)
    tr.append(trace_line)

for d in [0, 50, 100]:
    df1 = df_lake_days(d)
    trace = go.Scatter(
        x=df1.index,
        y=df1.day,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'Байкал, глубина={d}',
        error_y=dict(
            type='data',
            array=df1.next,
            arrayminus=df1.before,
            visible=True
        )
    )
    p = np.poly1d(np.polyfit(df1.index, df1.day, 1))
    trace_line = go.Scatter(
        x=np.linspace(min(df1.index), max(df1.index), 1000),
        y=p(np.linspace(min(df1.index), max(df1.index), 1000)),
        mode='lines',
        line=dict(color=colors[c_ind], width=2),
        name=f'Линейная аппроксимация для Байкала, глубина={d}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)
    tr.append(trace_line)

fig = go.Figure(data=tr)
fig.update_layout(
    title='Номер дня в году с максимальной температурой',
    xaxis_title='Год',
    yaxis_title='Номер дня в году',
    showlegend=True
)
plotly.offline.plot(fig, filename='Номер_дня_в_году_с_максимальной_температурой.html', show_link=False)

'Номер_дня_в_году_с_максимальной_температурой.html'

In [23]:
#Разницы между максимумами температур по годам
tr = []
colors = ['blue', 'red', 'black', 'green', 'yellow']
c_ind = 0
city = 'Иркутск'
d = 0
#d_other = 0
df1 = df_city_days(city)
#df1 = df_lake_days(d_other)
df2 = df_lake_days(d)
df1 = df1.rename(columns={'day': 'day1', 'temp': 'temp1'})
df2 = df2.rename(columns={'day': 'day2', 'temp': 'temp2'})
con = pd.concat([df1, df2], axis=1, join='inner')
con['differ'] = con.day2 - con.day1

trace = go.Scatter(
    x=con.index,
    y=con.differ,
    mode='markers',
    marker=dict(size=10, color=colors[c_ind]),
    name=f'Разницы между пиками температуры для {city} и Байкал, глубина={d}'
)
p = np.poly1d(np.polyfit(con.index, con.differ, 1))
trace_line = go.Scatter(
    x=np.linspace(min(con.index), max(con.index), 1000),
    y=p(np.linspace(min(con.index), max(con.index), 1000)),
    mode='lines',
    line=dict(color=colors[c_ind], width=2),
    name=f'Линейная аппроксимация'
)
c_ind += 1
tr.append(trace)
tr.append(trace_line)

fig = go.Figure(data=tr)
fig.update_layout(
    title=f'Количество дней между пиками температуры в году для {city} и Байкал, глубина={d}',
    xaxis_title='Год',
    yaxis_title='Количество дней между пиками температуры',
    showlegend=True
)
plotly.offline.plot(fig, filename='Количество_дней_между_пиками.html', show_link=False)

'Количество_дней_между_пиками.html'

In [24]:
#Максимумы температуры по годам для всех глубин и метеостанций
tr = []
colors = ['red', 'blue', 'black', 'green', 'yellow']
c_ind = 0

for city in ['Иркутск', 'Улан-Удэ', 'Большое Голоустное', 'Бабушкин', 'Чита']:
    df1 = df_city_days(city)
    trace = go.Scatter(
        x=df1.index,
        y=df1.temp,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'{city}'
    )
    p = np.poly1d(np.polyfit(df1.index, df1.temp, 1))
    trace_line = go.Scatter(
        x=np.linspace(min(df1.index), max(df1.index), 1000),
        y=p(np.linspace(min(df1.index), max(df1.index), 1000)),
        mode='lines',
        line=dict(color=colors[c_ind], width=2),
        name=f'Линейная аппроксимация для {city}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)
    tr.append(trace_line)

for d in [0, 50, 100]:
    df1 = df_lake_days(d)
    trace = go.Scatter(
        x=df1.index,
        y=df1.temp,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'Байкал, глубина={d}'
    )
    p = np.poly1d(np.polyfit(df1.index, df1.temp, 1))
    trace_line = go.Scatter(
        x=np.linspace(min(df1.index), max(df1.index), 1000),
        y=p(np.linspace(min(df1.index), max(df1.index), 1000)),
        mode='lines',
        line=dict(color=colors[c_ind], width=2),
        name=f'Линейная аппроксимация для Байкала, глубина={d}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)
    tr.append(trace_line)

fig = go.Figure(data=tr)
fig.update_layout(
    title='Максимумы температуры по годам',
    xaxis_title='Год',
    yaxis_title='Максимальная температура, °C',
    showlegend=True
)
plotly.offline.plot(fig, filename='Максимумы_температуры_по_годам.html', show_link=False)

'Максимумы_температуры_по_годам.html'

In [25]:
#Средние температуры по годам с месяца m1 по месяц m2 (теплый период) для всех глубин и метеостанций
tr = []
colors = ['red', 'blue', 'black', 'green', 'yellow']
c_ind = 0
m1 = 6
m2 = 11

for city in ['Иркутск', 'Улан-Удэ', 'Большое Голоустное', 'Бабушкин', 'Чита']:
    df1 = df_city_mean(city, m1, m2)
    trace = go.Scatter(
        x=df1.index,
        y=df1.temp,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'{city}'
    )
    p = np.poly1d(np.polyfit(df1.index, df1.temp, 1))
    trace_line = go.Scatter(
        x=np.linspace(min(df1.index), max(df1.index), 1000),
        y=p(np.linspace(min(df1.index), max(df1.index), 1000)),
        mode='lines',
        line=dict(color=colors[c_ind], width=2),
        name=f'Линейная аппроксимация для {city}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)
    tr.append(trace_line)

for d in [0, 50, 100]:
    df1 = df_lake_mean(d, m1, m2)
    trace = go.Scatter(
        x=df1.index,
        y=df1.temp,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'Байкал, глубина={d}'
    )
    p = np.poly1d(np.polyfit(df1.index, df1.temp, 1))
    trace_line = go.Scatter(
        x=np.linspace(min(df1.index), max(df1.index), 1000),
        y=p(np.linspace(min(df1.index), max(df1.index), 1000)),
        mode='lines',
        line=dict(color=colors[c_ind], width=2),
        name=f'Линейная аппроксимация для Байкала, глубина={d}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)
    tr.append(trace_line)

fig = go.Figure(data=tr)
fig.update_layout(
    title=f'Средние температуры по годам с месяца {m1} по месяц {m2} (теплый период)',
    xaxis_title='Год',
    yaxis_title='Средняя температура, °C',
    showlegend=True
)
plotly.offline.plot(fig, filename='Средние_температуры_по_годам.html', show_link=False)

'Средние_температуры_по_годам.html'